# Análisis de Experimento A/B

En esta parte del proyecto, se realizará la validación estadística de las pruebas A/B realizadas por Rappiplus

In [2]:
# Importación de librerías
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency # Prueba si dos variables cat son independientes entre si
from scipy.stats import ttest_ind # Compara promedios de dos grupos estadisticamente distintos
from scipy.stats import levene # Prueba previa a ttest, prueba si la varianza es similar en ambos o no)
from statsmodels.stats.proportion import proportions_ztest # Compara proporciones entre dos grupos.

In [3]:
test = pd.read_csv(r'../data/experiment_checkout_ui.csv')

## Exploración inicial

In [4]:
print(f'Cantidad de registros: {test.shape[0]}')

Cantidad de registros: 10000


In [5]:
test.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [6]:
# Comparación del tiempo de duración promedio por sesión en cada variante
test.groupby('variante')['duracion_sesion'].mean().round(2)

variante
control        161.04
tratamiento    158.70
Name: duracion_sesion, dtype: float64

In [7]:
# Porcentaje de la participación de los dispositivos por variante
(test.groupby('variante')['dispositivo'].value_counts(normalize= True)*100).round(2)

variante     dispositivo
control      desktop        50.51
             mobile         49.49
tratamiento  desktop        50.33
             mobile         49.67
Name: proportion, dtype: float64

In [8]:
# Porcentaje de participación de países por variante
(test.groupby('variante')['pais'].value_counts(normalize= True)*100).round(2)

variante     pais     
control      Mexico       33.78
             Argentina    33.35
             Colombia     32.87
tratamiento  Mexico       34.32
             Argentina    32.99
             Colombia     32.69
Name: proportion, dtype: float64

Se verificó el balance de las variables categóricas (dispositivo, país) y numéricas (duración de sesión) relevantes entre los grupos control y tratamiento, confirmando que la aleatorización fue equilibrada y que ambos grupos son comparables. Esto permite continuar con el análisis de conversión con confianza en la validez del experimento.

## Análisis comparativo entre control y tratamiento

Hipótesis estadística

- **H0 (Hipótesis nula):** El cambio de diseño en el checkout (grupo tratamiento) no tiene un efecto significativo sobre el comportamiento de los usuarios (tasa de conversión y duración de sesión) en comparación con el diseño actual (grupo control).
- **H1 (Hipótesis alternativa):** El cambio de diseño en el checkout (grupo tratamiento) sí tiene un efecto significativo sobre el comportamiento de los usuarios (tasa de conversión y/o duración de sesión) en comparación con el diseño actual (grupo control).

**Test estadístico:** Prueba z de proporciones (para la tasa de conversión) y prueba t de Student para muestras independientes (para la duración de sesión)
**Nivel de significancia alpha:** 0.05

### Verificación de conversión entre variante

In [9]:
# Identificación de valores necesarios (Cantidad de clientes por grupo y cantidad de conversión)
conversion = test.groupby('variante')['convirtio'].sum() # Agrupación por página y por conversión
totales = test.groupby('variante')['convirtio'].count() # Cuántos usuarios de cada página han comprado
exitos = [conversion['control'], conversion['tratamiento']] # Desglose de conversiones por grupo
observaciones = [totales['control'], totales['tratamiento']] # Cuántas son las observaciones totales de los grupos

print(f'Conversiones en Control: {exitos[0]} \nConversiones en Tratamiento: {exitos[1]}')
print() # Salto de línea
print(f'Observaciones en Control: {observaciones[0]} \nObservaciones en Tratamiento: {observaciones[1]}')

Conversiones en Control: 779 
Conversiones en Tratamiento: 820

Observaciones en Control: 4965 
Observaciones en Tratamiento: 5035


In [10]:
# z-test
z_stat, p_value = proportions_ztest(exitos, observaciones)

print(f'Estadístico Z: {z_stat}')
print(f'Valor P: {p_value}')

alpha = 0.05 # umbral de significancia
if p_value < alpha:
    print('Se rechaza la hipótesis nula: Hay evidencia de una diferencia')
else:
    print('No se rechaza hipótesis nula: No tenemos suficiente evidencia de una diferencia')

tasa_control = exitos[0] / observaciones[0]
tasa_tratamiento= exitos[1] / observaciones[1]

print(f'\nTasa de conversión Control: {tasa_control:.2%} \nTasa de conversión Tratamiento : {tasa_tratamiento:.2%}')

Estadístico Z: -0.8132782986429474
Valor P: 0.41605851639119995
No se rechaza hipótesis nula: No tenemos suficiente evidencia de una diferencia

Tasa de conversión Control: 15.69% 
Tasa de conversión Tratamiento : 16.29%


### Verificación de duración de sesión por variante

In [11]:
# Tiempo de duración por variante
duracion_control = test[(test['variante'] == 'control') & (test['duracion_sesion'])]
duracion_tratamiento = test[(test['variante'] == 'tratamiento') & (test['duracion_sesion'])]

# Verificación de cantiad de datos por grupo
print(f'Duración en Control: {len(duracion_control)}')
print(f'Duración en Tratamiento: {len(duracion_tratamiento)}')
print(f'Total: {len(duracion_control) + len(duracion_tratamiento)}, un {(len(duracion_control) + len(duracion_tratamiento)) / len(test) *100}% de todos los registros')

Duración en Control: 4965
Duración en Tratamiento: 5035
Total: 10000, un 100.0% de todos los registros


In [12]:
# Prueba de varianzas iguales
l_stat, p_value_var = levene(duracion_control['duracion_sesion'], 
                             duracion_tratamiento['duracion_sesion'])

print(f'Estadístico Levene: {l_stat}')
print(f'Valor p: {p_value_var}')
print()
alpha = 0.05 # Umbral de significacia
if p_value_var < alpha:
    print('Rechazamos la hipótesis nula: Hay evidencia de varianzas diferentes (equal_var = False)')
else:
    print('No rechazamos la hipótesis nula: No hay evidencia de varianzas diferentes (equal_var = True)')

Estadístico Levene: 0.6403307695734209
Valor p: 0.42361005887014236

No rechazamos la hipótesis nula: No hay evidencia de varianzas diferentes (equal_var = True)


In [13]:
# t_test
t_stat, p_value = ttest_ind(duracion_control['duracion_sesion'],
                            duracion_tratamiento['duracion_sesion'],
                            equal_var= True)
print(f'Estadístico t: {t_stat} \nValor P: {p_value}')

print() # Salto de línea

# Umbral de significancia
if p_value < alpha:
    print('Rechazamos la hipótesis nula: Hay evidencia de diferencias')
else:
    print('No rechazamos la hipótesis nula: No hay evidencia de diferencias')

print() # Salto de línea

# Cálculo de promedios
mean_duracion_control= duracion_control['duracion_sesion'].mean()
mean_duracion_tratamiento= duracion_tratamiento['duracion_sesion'].mean()
print(f'Promedio de duración de sesión en control: {mean_duracion_control:.4} \nPromedio de duración de sesión en tratamiento: {mean_duracion_tratamiento:.04}')
diferencia = mean_duracion_control - mean_duracion_tratamiento
print(f'Diferencia en duración promedio: {diferencia:.2f} segundos más por usuario en Control')

Estadístico t: 1.4454605605858548 
Valor P: 0.1483599058199363

No rechazamos la hipótesis nula: No hay evidencia de diferencias

Promedio de duración de sesión en control: 161.0 
Promedio de duración de sesión en tratamiento: 158.7
Diferencia en duración promedio: 2.34 segundos más por usuario en Control


### Homogeneidad de efectos por segmento

In [20]:
def test_conversion(df, col_segmento):
    resultados=[]

    for valor in df[col_segmento].unique():
        subset = df[df[col_segmento] == valor]

        conversion = subset.groupby('variante')['convirtio'].sum()
        observaciones = subset.groupby('variante')['convirtio'].count()

        exitos = [conversion['control'], conversion['tratamiento']]
        totales = [observaciones['control'], observaciones['tratamiento']]

        z_stat, p_value = proportions_ztest(exitos, totales)

        resultados.append({
            col_segmento: valor,
            'Tasa_control': round(exitos[0] / totales[0] * 100, 2),
            'tasa_tratamiento' : round(exitos[1] / totales[1] * 100, 2),
            'z_stat' : round(z_stat, 4),
            'Valor_p': round(p_value, 4),
            'Significativo' : p_value < 0.05
        })
    return pd.DataFrame(resultados)

In [22]:
resultado_dispositivo = test_conversion(test, 'dispositivo')
resultado_pais = test_conversion(test, 'pais')

In [23]:
resultado_dispositivo

,dispositivo,Tasa_control,tasa_tratamiento,z_stat,Valor_p,Significativo
0,mobile,12.94,14.35,-1.4473,0.1478,False
1,desktop,18.38,18.19,0.1732,0.8625,False


In [24]:
resultado_pais

,pais,Tasa_control,tasa_tratamiento,z_stat,Valor_p,Significativo
0,Argentina,16.18,16.44,-0.1967,0.8441,False
1,Mexico,15.03,16.90,-1.4900,0.1362,False
2,Colombia,15.87,15.49,0.2976,0.7660,False


## Conclusión del análisis comparativo

El cambio de UI en el checkout no mostró un impacto estadísticamente significativo en la tasa de conversión (p=0.416) ni en la duración de sesión (p=0.148), tanto a nivel global como al segmentar por dispositivo y país (todos los p-values > 0.05). Ningún subgrupo mostró evidencia sólida de un efecto oculto, las diferencias más cercanas a significancia (mobile, México) son consistentes con variación aleatoria dado el tamaño de muestra reducido al segmentar. Con la evidencia disponible, no se recomienda atribuir cambios en el negocio a esta modificación de UI; de considerarse relevante seguir explorando el diseño, sería necesario un experimento con mayor tamaño de muestra para detectar efectos pequeños con suficiente poder estadístico.